# Research Librarian

**Project 15** — Core RAG Projects

Build a system that indexes multiple research papers and answers research questions
with source-backed evidence drawn from more than one paper. Learn how multi-hop
retrieval works — finding information that spans multiple documents to compose
a complete answer.

**Key Skills:** Multi-document indexing, multi-hop retrieval, cross-paper evidence
synthesis, citation tracking, research question decomposition.

## Project Overview

Research involves reading dozens of papers and connecting ideas across them.
A single paper rarely answers a complex question fully. Researchers need to:

1. **Find relevant passages** across a large corpus of papers
2. **Connect evidence** from multiple sources to answer compound questions
3. **Track citations** so every claim links back to its source
4. **Distinguish consensus from contradiction** when papers disagree

This notebook builds a **Research Librarian** that:
- Indexes a corpus of 5 research paper abstracts + body sections
- Answers single-hop questions (one paper suffices)
- Answers **multi-hop questions** (requires evidence from 2+ papers)
- Provides full source citations for every part of the answer
- Detects when papers support vs contradict each other

### What is Multi-Hop Retrieval?

**Single-hop**: "What accuracy did BERT achieve on GLUE?" → one paper answers this directly.

**Multi-hop**: "How do transformer-based models compare to CNN-based models for text
classification?" → requires retrieving from a BERT paper AND a CNN paper, then
synthesizing the comparison.

Multi-hop is harder because:
- No single chunk contains the full answer
- The retriever must find semantically related but distinct evidence
- The system must combine information logically, not just concatenate text

## Learning Goals

By the end of this notebook you will understand:
- How to index and retrieve from a multi-paper corpus
- The difference between single-hop and multi-hop retrieval
- How query decomposition breaks complex questions into retrievable sub-queries
- How to synthesize evidence from multiple papers with proper citations
- How to detect agreement, contradiction, and complementary evidence
- The limitations of multi-hop RAG without a reasoning LLM

## Problem Statement

**Scenario:** A graduate student is preparing a literature review on **"Modern
Approaches to Natural Language Understanding."** They have collected 5
representative papers covering different NLP techniques. They need to:

- Quickly find what each paper contributes to specific sub-topics
- Answer questions that require combining findings from multiple papers
- Identify where papers agree (consensus) or disagree (controversy)
- Build a structured evidence map with full citations

**Goal:** Build a Research Librarian that answers both simple and multi-hop
research questions with properly cited, multi-source evidence.

## Why This Project Matters

| Challenge | How Multi-Hop RAG Helps |
|-----------|------------------------|
| Papers are long and dense | Retrieval finds specific relevant passages |
| Answers span multiple papers | Multi-hop retrieval combines evidence |
| Claims need attribution | Citation tracking links every fact to its source |
| Contradictions hide in literature | Cross-paper comparison surfaces disagreements |
| Literature reviews are time-intensive | Automated evidence gathering accelerates research |

### Real-world applications
- **Literature reviews** — automated evidence gathering across a field
- **Systematic reviews** — structured comparison of study results
- **Grant writing** — quickly finding supporting evidence for proposals
- **Peer review** — checking claims against cited literature
- **Research assistants** — helping domain experts explore unfamiliar sub-fields

## Environment Setup

In [1]:
import os
import re
import json
import hashlib
import textwrap
import warnings
from collections import defaultdict, Counter
from datetime import datetime
from itertools import combinations

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_ST = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available — using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available — using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


In [2]:
# ── Configuration ──
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
TOP_K = 5
MULTI_HOP_TOP_K = 3        # per sub-query
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))

print(f"Chunk size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
print(f"Top-K: {TOP_K}, Multi-hop per sub-query: {MULTI_HOP_TOP_K}")

Chunk size: 500, Overlap: 80
Top-K: 5, Multi-hop per sub-query: 3


## Research Paper Corpus

We create 5 fictional but realistic NLP research papers. Each paper has:
- **Metadata** (title, authors, year, venue, DOI-style ID)
- **Abstract** — concise summary
- **Multiple sections** — introduction, method, experiments, results, discussion

This simulates a real literature corpus where answers to complex questions
are distributed across multiple papers.

In [3]:
# ── Research papers ──

PAPERS = [
    {
        "paper_id": "P1",
        "title": "Contextual Word Representations via Transformer Pre-training",
        "authors": ["Chen, W.", "Liu, R.", "Patel, S."],
        "year": 2023,
        "venue": "ACL 2023",
        "doi": "10.1234/acl.2023.001",
        "sections": [
            {
                "heading": "Abstract",
                "text": """We present TransContext, a transformer-based language model pre-trained on 800GB of web text using masked language modeling and next sentence prediction objectives. TransContext achieves state-of-the-art results on 8 out of 11 NLU benchmarks, including GLUE (92.4%), SuperGLUE (89.1%), and SQuAD 2.0 (F1: 93.2%). Our key innovation is a dynamic masking schedule that adapts the masking ratio during training based on the model's perplexity, leading to 15% faster convergence compared to static masking approaches like BERT. We release models in Base (110M parameters) and Large (340M parameters) configurations."""
            },
            {
                "heading": "Introduction",
                "text": """Pre-trained language models have revolutionized natural language processing. Since the introduction of BERT by Devlin et al. (2019), the NLP community has seen rapid advances in transfer learning approaches. However, current pre-training strategies use static masking ratios throughout training, which we hypothesize leads to suboptimal learning dynamics. Early in training, the model benefits from easier examples (lower masking), while later stages benefit from harder examples (higher masking). We propose dynamic masking scheduling, which adjusts the masking ratio from 10% to 25% over the course of training based on validation perplexity. Our approach builds on the transformer architecture with 12 attention heads and a hidden dimension of 768 for the Base model."""
            },
            {
                "heading": "Methodology",
                "text": """TransContext uses the standard transformer encoder architecture. Pre-training uses two objectives: (1) Masked Language Modeling (MLM) with our novel dynamic masking schedule, and (2) Next Sentence Prediction (NSP). The dynamic masking schedule starts at 10% masking ratio and increases linearly to 25% over 500K training steps, with adjustments based on a moving average of validation perplexity computed every 10K steps. If perplexity plateaus for 3 consecutive checkpoints, the masking ratio increases by 2%. Training data consists of English Wikipedia, BookCorpus, and CommonCrawl (filtered), totaling 800GB. We use the Adam optimizer with learning rate 1e-4, warmup over 10K steps, and linear decay. Training was performed on 64 A100 GPUs for 14 days."""
            },
            {
                "heading": "Results",
                "text": """On GLUE benchmark, TransContext-Large achieves an average score of 92.4%, compared to 87.6% for BERT-Large and 90.1% for RoBERTa-Large. On SuperGLUE, we achieve 89.1% versus BERT's 84.3%. On SQuAD 2.0, our model achieves F1 of 93.2%, a 2.1 point improvement over BERT-Large. Ablation studies show that dynamic masking contributes approximately 1.8 points of improvement on GLUE, while the remaining gains come from larger training data. The Base model (110M parameters) achieves 89.7% on GLUE, demonstrating that dynamic masking benefits smaller models proportionally more than larger ones."""
            },
            {
                "heading": "Discussion",
                "text": """Our results demonstrate that training dynamics matter as much as model architecture for pre-trained language models. The curriculum-style approach of dynamic masking mimics how humans learn — starting with easier patterns before tackling complex ones. One limitation is the computational overhead of periodic perplexity evaluation, which adds approximately 5% to total training time. Compared to CNN-based approaches for text classification (e.g., TextCNN achieving 85% on SST-2), transformer-based pre-training shows clear advantages for tasks requiring long-range dependencies, though CNNs remain competitive for simple classification tasks with shorter texts. Future work will explore extending dynamic masking to generative pre-training objectives."""
            },
        ]
    },
    {
        "paper_id": "P2",
        "title": "Efficient Fine-Tuning with Adapter Layers for NLU Tasks",
        "authors": ["Martinez, A.", "Kim, J.", "Thompson, E."],
        "year": 2023,
        "venue": "EMNLP 2023",
        "doi": "10.1234/emnlp.2023.042",
        "sections": [
            {
                "heading": "Abstract",
                "text": """Full fine-tuning of large language models is computationally expensive and risks catastrophic forgetting. We propose AdaptNLU, a parameter-efficient fine-tuning method that inserts lightweight adapter layers between transformer blocks. Each adapter adds only 3.6% additional parameters while achieving 97% of full fine-tuning performance on GLUE (90.8% vs 92.1% for full fine-tuning of a 340M parameter model). Our adapters reduce GPU memory requirements by 60% and training time by 45%, making state-of-the-art NLU accessible to researchers with limited compute budgets. We also introduce task-specific adapter routing, which learns to activate different adapter subsets for different downstream tasks."""
            },
            {
                "heading": "Introduction",
                "text": """The trend toward larger pre-trained models creates a tension: bigger models achieve better performance but are increasingly expensive to fine-tune. A 340M parameter model requires 16GB of GPU memory for inference alone and over 40GB for full gradient computation during fine-tuning. This puts state-of-the-art NLP beyond the reach of many research groups. Adapter-based approaches offer a middle ground. By freezing the pre-trained weights and inserting small trainable modules, we can achieve strong performance while updating fewer than 5% of the total parameters. Our work builds on Houlsby et al. (2019) but introduces task-aware routing that shares adapter parameters across related tasks."""
            },
            {
                "heading": "Methodology",
                "text": """AdaptNLU inserts adapter modules after the feed-forward layer in each transformer block. Each adapter consists of a down-projection (768 to 64 dimensions), a non-linear activation (GELU), and an up-projection (64 to 768 dimensions), with a residual connection. Total additional parameters per adapter: 98,304. For a 12-layer model, the total adapter parameters are 1.18M, compared to 110M base parameters (1.07% overhead for Base, 3.6% for our recommended configuration with wider bottleneck). During fine-tuning, all base model parameters are frozen; only adapter parameters and the classification head are updated. Task-specific routing uses a lightweight attention mechanism that scores each adapter's relevance for the current input based on a learned task embedding. We train on GLUE, SuperGLUE, and SQuAD 2.0 using the same pre-trained backbone as TransContext-Base."""
            },
            {
                "heading": "Results",
                "text": """AdaptNLU achieves 90.8% on GLUE with only 3.6% additional parameters, compared to 92.1% for full fine-tuning (97.2% relative performance). On SuperGLUE, adapters achieve 86.5% vs 88.3% for full fine-tuning. Memory usage during training drops from 42GB to 16.8GB (60% reduction). Training time decreases from 8 hours to 4.4 hours for GLUE (45% reduction) on a single A100 GPU. Task-specific routing improves multi-task performance by 1.2 points over static adapters, suggesting that selective adapter activation captures task-specific features better. Comparison with LoRA: AdaptNLU outperforms LoRA (rank=8) by 0.6 points on GLUE while using similar parameter counts, likely due to the non-linear bottleneck in adapters vs LoRA's linear projections."""
            },
            {
                "heading": "Discussion",
                "text": """Parameter-efficient fine-tuning democratizes access to large language models. Our adapters achieve near-full-fine-tuning performance with dramatically reduced cost. The task routing mechanism is particularly valuable for multi-task deployments, where a single base model can serve multiple downstream applications. Limitations: adapter performance degrades more on tasks that differ significantly from the pre-training domain (e.g., biomedical NER drops to 89% of full fine-tuning performance). This suggests that adapters work best when the pre-training data distribution covers the downstream domain. Compared to prompt tuning approaches, adapters provide more stable training and better performance on smaller datasets (under 10K examples), likely because adapters modify intermediate representations while prompt tuning only modifies the input."""
            },
        ]
    },
    {
        "paper_id": "P3",
        "title": "Cross-Lingual Transfer with Multilingual Transformers",
        "authors": ["Johansson, L.", "Yamamoto, T.", "Okafor, C."],
        "year": 2024,
        "venue": "NAACL 2024",
        "doi": "10.1234/naacl.2024.015",
        "sections": [
            {
                "heading": "Abstract",
                "text": """We study zero-shot cross-lingual transfer using multilingual transformer models. Our model, PolyGlot-XL, is pre-trained on 100 languages using a combination of masked language modeling and translation language modeling (TLM). On the XTREME benchmark, PolyGlot-XL achieves an average score of 82.3% across 40 languages, outperforming XLM-RoBERTa by 3.1 points. We find that languages with less than 100MB of pre-training data benefit most from cross-lingual transfer, with low-resource language performance improving by up to 12 points when paired with related high-resource languages during TLM pre-training."""
            },
            {
                "heading": "Introduction",
                "text": """Most NLP research focuses on English, leaving thousands of languages underserved. Multilingual models offer a path to NLP capabilities for low-resource languages through cross-lingual transfer — training on resource-rich languages and evaluating on others. The key insight is that multilingual transformers develop shared representations across languages, enabling zero-shot transfer. XLM-RoBERTa (Conneau et al., 2020) demonstrated this at scale, but performance gaps between high-resource and low-resource languages persist. We hypothesize that strategic use of translation language modeling (TLM), where the model sees parallel sentences in two languages simultaneously, creates stronger cross-lingual bridges, especially for distant language pairs."""
            },
            {
                "heading": "Methodology",
                "text": """PolyGlot-XL uses the transformer encoder architecture with 24 layers, 16 attention heads, and hidden dimension 1024 (550M parameters). Pre-training objectives: (1) Masked Language Modeling on monolingual data in 100 languages; (2) Translation Language Modeling on 50 language pairs using parallel corpora from OPUS and CCMatrix. Total pre-training data: 2.5TB. The TLM objective concatenates parallel sentences and randomly masks tokens in either language, forcing the model to use cross-lingual context. We introduce language-family-aware sampling: during TLM, we pair languages within the same family (e.g., Spanish-Portuguese, Japanese-Korean) 60% of the time and across families 40% of the time. Training: 128 A100 GPUs for 21 days. Vocabulary: 250K tokens using SentencePiece with language-balanced sampling."""
            },
            {
                "heading": "Results",
                "text": """On XTREME, PolyGlot-XL achieves 82.3% average across all 40 languages, compared to 79.2% for XLM-RoBERTa-Large. Gains are largest for low-resource languages: Swahili (+8.4 points), Burmese (+11.7 points), and Quechua (+12.1 points). For high-resource languages (English, Chinese, Spanish), improvements are modest (1-2 points), suggesting diminishing returns for well-resourced languages. Zero-shot cross-lingual NER achieves F1 of 74.2% averaged across 40 languages (trained only on English). On cross-lingual question answering (XQuAD), we achieve EM of 72.8%, a 4.3 point improvement over XLM-R. Language-family-aware TLM sampling contributes 2.1 points of the total improvement; removing it reduces XTREME score to 80.2%."""
            },
            {
                "heading": "Discussion",
                "text": """Our results confirm that strategic pre-training choices — particularly TLM with language-family awareness — significantly improve cross-lingual transfer. The largest gains for low-resource languages suggest that these languages benefit most from cross-lingual bridging. However, we observe that agglutinative languages (Turkish, Finnish) still lag behind analytical languages (English, Chinese) by 5-8 points, likely due to subword tokenization challenges. Compared to translation-based approaches (translate-test), zero-shot transfer is faster and avoids translation errors but sacrifices approximately 3-5 points of accuracy for distant language pairs. Future work will explore conditional computation to reduce inference cost, as the 550M parameter model is expensive for deployment in low-resource settings where compute is also limited."""
            },
        ]
    },
    {
        "paper_id": "P4",
        "title": "Robust NLU Under Distribution Shift: Adversarial and Domain Adaptation",
        "authors": ["Park, S.", "Gonzalez, M.", "Ivanova, E."],
        "year": 2024,
        "venue": "ICLR 2024",
        "doi": "10.1234/iclr.2024.078",
        "sections": [
            {
                "heading": "Abstract",
                "text": """Pre-trained models achieve impressive benchmark scores but often fail on out-of-distribution (OOD) data. We present RobustNLU, a framework for improving model robustness through adversarial training and unsupervised domain adaptation. On our new benchmark, NLU-Shift (containing 12 distribution shift scenarios), RobustNLU maintains 91% of in-distribution performance under shift, compared to 76% for standard fine-tuning. Key innovations: (1) gradient-based adversarial example generation during fine-tuning, (2) domain-adversarial training that learns domain-invariant representations, and (3) a confidence calibration method that provides reliable uncertainty estimates under distribution shift."""
            },
            {
                "heading": "Introduction",
                "text": """The impressive benchmark results of pre-trained language models mask a fragility problem: these models are surprisingly brittle when input distribution changes. A sentiment classifier trained on product reviews may fail on social media text. A question answering model trained on Wikipedia may struggle with scientific papers. This distribution shift problem affects real-world deployment, where training and test distributions inevitably differ. Prior work on adversarial training (Madry et al., 2018) and domain adaptation (Ganin et al., 2016) has shown promise individually, but few studies combine these approaches for NLU tasks. We propose an integrated framework that addresses three aspects of robustness: adversarial robustness (resistance to input perturbations), domain robustness (generalization across domains), and calibration (reliable confidence estimates)."""
            },
            {
                "heading": "Methodology",
                "text": """RobustNLU extends the standard fine-tuning pipeline with three components: (1) Adversarial Training: We generate adversarial examples using projected gradient descent (PGD) on the embedding space. At each training step, we compute the worst-case perturbation within an epsilon-ball (epsilon=0.01 in embedding space) and train the model on both clean and adversarial examples. (2) Domain-Adversarial Training: We add a domain classifier head with gradient reversal. The model learns features that are discriminative for the NLU task but invariant to the domain, using the same approach as DANN (Ganin et al., 2016) adapted for transformer architectures. (3) Confidence Calibration: We apply temperature scaling post-training and add an auxiliary calibration loss during training. The calibration loss penalizes the gap between predicted confidence and actual accuracy on a held-out calibration set. All three components are applied to a 340M parameter transformer backbone (same architecture as TransContext-Large)."""
            },
            {
                "heading": "Results",
                "text": """We evaluate on NLU-Shift, our benchmark with 12 distribution shift scenarios across 4 tasks (sentiment analysis, NLI, QA, NER). Results: Standard fine-tuning retains 76% of in-distribution performance under shift. Adversarial training alone improves retention to 83%. Domain adaptation alone improves to 85%. Combined (RobustNLU) achieves 91% retention. On adversarial benchmarks (AdvGLUE), RobustNLU achieves 78.3% accuracy compared to 64.1% for standard fine-tuning — a 14.2 point improvement. Calibration: Expected Calibration Error (ECE) drops from 12.3% (standard) to 3.8% (RobustNLU). In-distribution performance: RobustNLU sacrifices only 0.8 points on GLUE (91.3% vs 92.1% for standard fine-tuning), a favorable tradeoff given the robustness gains. Training cost: adversarial training adds 40% overhead; domain adaptation adds 15%; total overhead is approximately 60%."""
            },
            {
                "heading": "Discussion",
                "text": """Our results show that robustness can be achieved with minimal in-distribution performance sacrifice. The key insight is that adversarial training and domain adaptation are complementary: adversarial training improves local robustness (resistance to small perturbations) while domain adaptation improves global robustness (generalization across domains). The calibration component is critical for deployment: a model that knows when it doesn't know is more useful than one that is confidently wrong. Comparison with data augmentation baselines: random text augmentation (EDA, Wei & Zou 2019) achieves 80% distribution shift retention, compared to our 91%, suggesting that targeted adversarial perturbations are more effective than random augmentations. Limitations: RobustNLU primarily addresses surface-level distribution shifts. It does not address deep semantic shifts (e.g., temporal concept drift) or label distribution changes. The 60% training overhead is also a practical concern for very large models."""
            },
        ]
    },
    {
        "paper_id": "P5",
        "title": "Knowledge-Enhanced Pre-training for Commonsense NLU",
        "authors": ["Wang, H.", "Petrov, N.", "Al-Rashid, F."],
        "year": 2024,
        "venue": "ACL 2024",
        "doi": "10.1234/acl.2024.033",
        "sections": [
            {
                "heading": "Abstract",
                "text": """Standard pre-trained language models struggle with tasks requiring commonsense reasoning, such as physical plausibility, temporal reasoning, and causal inference. We present CommonsenseTransformer (CST), which augments transformer pre-training with structured knowledge from ConceptNet and ATOMIC knowledge graphs. CST achieves 84.7% on the CommonsenseQA benchmark (vs 72.1% for BERT-Large and 78.9% for RoBERTa-Large) and 81.2% on WinoGrande (vs 72.4% for BERT-Large). Our approach introduces a knowledge-attention mechanism that attends to relevant knowledge graph triples during pre-training, creating representations that encode both linguistic and commonsense knowledge."""
            },
            {
                "heading": "Introduction",
                "text": """Despite achieving high scores on many NLU benchmarks, pre-trained language models often fail at tasks requiring basic commonsense. Examples: 'A man put cheese in the refrigerator. The next day, was the cheese fresh or rotten?' requires understanding of food preservation. 'She poured water from the bottle into the cup until it was full. What was full?' requires physical reasoning about containers. These failures arise because commonsense knowledge is implicit in text and rarely stated explicitly. Knowledge graphs like ConceptNet contain structured commonsense facts (e.g., 'refrigerator' UsedFor 'preserving food'), but integrating this structured knowledge with unstructured pre-training remains challenging."""
            },
            {
                "heading": "Methodology",
                "text": """CST modifies the transformer architecture by adding a knowledge-attention layer after every 4th self-attention layer. The knowledge-attention mechanism works as follows: (1) For each input token, we retrieve the top-10 relevant triples from ConceptNet using entity linking and embedding similarity. (2) Each triple (head, relation, tail) is encoded as a fixed-length vector using a learned triple encoder. (3) The knowledge-attention layer attends to these triple embeddings using cross-attention, with the token representations as queries and triple embeddings as keys and values. (4) The output is added to the token representations via a gated residual connection. Pre-training data: same as TransContext (800GB of web text) plus 3.4M triples from ConceptNet and 877K events from ATOMIC. Total model parameters: 380M (340M base + 40M knowledge modules). Training: 64 A100 GPUs for 18 days, approximately 30% more compute than standard pre-training."""
            },
            {
                "heading": "Results",
                "text": """CommonsenseQA: CST achieves 84.7%, compared to 72.1% (BERT-Large), 78.9% (RoBERTa-Large), and 80.3% (our TransContext baseline). WinoGrande: 81.2% vs 72.4% (BERT) and 77.1% (RoBERTa). PIQA (physical commonsense): 86.3% vs 77.8% (BERT). On standard NLU (GLUE), CST performs comparably to TransContext (92.0% vs 92.4%), showing that knowledge enhancement does not harm standard NLU. Ablation: Knowledge-attention alone (without ATOMIC) achieves 82.1% on CommonsenseQA. Adding ATOMIC improves to 84.7%, confirming that event-based commonsense is complementary to entity-based ConceptNet knowledge. Analysis of attention patterns shows that knowledge-attention heads consistently attend to relevant triples: for temporal questions, they attend to temporal relations; for causal questions, to causal relations."""
            },
            {
                "heading": "Discussion",
                "text": """CST demonstrates that explicit knowledge integration substantially improves commonsense reasoning without degrading standard NLU performance. The gated residual connection is crucial — without it, knowledge information overwhelms the pre-trained representations, hurting GLUE by 1.5 points. Comparison with other knowledge-enhanced models: ERNIE (Zhang et al., 2019) achieves 78.2% on CommonsenseQA by integrating knowledge during pre-training through entity embeddings. Our knowledge-attention mechanism is more flexible because it dynamically selects relevant triples rather than using fixed entity embeddings. Limitations: (1) Knowledge coverage: ConceptNet and ATOMIC have limited coverage of specialized domains (medical, legal). (2) Entity linking errors propagate — approximately 8% of retrieved triples are irrelevant, adding noise. (3) The 30% compute overhead and extra 40M parameters increase deployment costs. Future work will explore dynamic knowledge graph construction from text and multi-hop reasoning over knowledge graph paths."""
            },
        ]
    },
]

print(f"Created corpus of {len(PAPERS)} research papers:\n")
for paper in PAPERS:
    total_chars = sum(len(s["text"]) for s in paper["sections"])
    print(f"  [{paper['paper_id']}] {paper['title']}")
    print(f"      {', '.join(paper['authors'])} | {paper['venue']}")
    print(f"      Sections: {len(paper['sections'])} | {total_chars:,} chars")


Created corpus of 5 research papers:

  [P1] Contextual Word Representations via Transformer Pre-training
      Chen, W., Liu, R., Patel, S. | ACL 2023
      Sections: 5 | 3,478 chars
  [P2] Efficient Fine-Tuning with Adapter Layers for NLU Tasks
      Martinez, A., Kim, J., Thompson, E. | EMNLP 2023
      Sections: 5 | 3,866 chars
  [P3] Cross-Lingual Transfer with Multilingual Transformers
      Johansson, L., Yamamoto, T., Okafor, C. | NAACL 2024
      Sections: 5 | 3,739 chars
  [P4] Robust NLU Under Distribution Shift: Adversarial and Domain Adaptation
      Park, S., Gonzalez, M., Ivanova, E. | ICLR 2024
      Sections: 5 | 4,469 chars
  [P5] Knowledge-Enhanced Pre-training for Commonsense NLU
      Wang, H., Petrov, N., Al-Rashid, F. | ACL 2024
      Sections: 5 | 4,190 chars


## Understanding Multi-Hop Retrieval

### Single-Hop vs Multi-Hop

**Single-hop retrieval** finds one passage that directly answers the question:
- Q: "What is the architecture of TransContext?"
- A single chunk from Paper P1 contains the full answer.

**Multi-hop retrieval** requires combining evidence from multiple passages or documents:
- Q: "How do parameter-efficient methods compare to full pre-training for commonsense tasks?"
- This requires: (1) Finding adapter/parameter-efficient results from **P2**,
  (2) Finding commonsense pre-training results from **P5**,
  (3) Synthesizing a comparison.

### Query Decomposition

The key technique for multi-hop is **query decomposition** — breaking a complex
question into simpler sub-queries that can each be answered by single-hop retrieval:

```
Complex: "How do different pre-training approaches handle distribution shift
          in multilingual settings?"
    ↓ decompose
Sub-query 1: "Pre-training approaches for distribution shift" → finds P4
Sub-query 2: "Multilingual pre-training approaches" → finds P3
Sub-query 3: "Distribution shift in multilingual models" → finds P3, P4
    ↓ synthesize
Combined answer with evidence from both papers
```

### Evidence Types in Multi-Hop

When combining evidence from multiple papers, we can identify:
- **Corroborating** — papers support the same conclusion with different evidence
- **Complementary** — papers address different aspects of the same topic
- **Contradicting** — papers disagree on a finding or interpretation

In [4]:
# ── Parse papers into sections ──

class PaperSection:
    """A section from a research paper with full metadata."""

    def __init__(self, paper, section):
        self.paper_id = paper["paper_id"]
        self.title = paper["title"]
        self.authors = paper["authors"]
        self.year = paper["year"]
        self.venue = paper["venue"]
        self.doi = paper["doi"]
        self.heading = section["heading"]
        self.text = section["text"].strip()

    @property
    def citation(self):
        first_author = self.authors[0].split(",")[0]
        return f"{first_author} et al. ({self.year})"

    @property
    def full_citation(self):
        author_str = ", ".join(self.authors)
        return f'{author_str}. "{self.title}." {self.venue}. DOI: {self.doi}'

    @property
    def metadata(self):
        return {
            "paper_id": self.paper_id,
            "title": self.title,
            "authors": ", ".join(self.authors),
            "year": str(self.year),
            "venue": self.venue,
            "heading": self.heading,
        }


all_sections = []
for paper in PAPERS:
    for section in paper["sections"]:
        ps = PaperSection(paper, section)
        all_sections.append(ps)

print(f"Parsed {len(all_sections)} sections from {len(PAPERS)} papers:\n")
for sec in all_sections:
    print(f"  [{sec.paper_id}] {sec.heading}: {len(sec.text)} chars — {sec.citation}")

Parsed 25 sections from 5 papers:

  [P1] Abstract: 611 chars — Chen et al. (2023)
  [P1] Introduction: 770 chars — Chen et al. (2023)
  [P1] Methodology: 755 chars — Chen et al. (2023)
  [P1] Results: 590 chars — Chen et al. (2023)
  [P1] Discussion: 752 chars — Chen et al. (2023)
  [P2] Abstract: 703 chars — Martinez et al. (2023)
  [P2] Introduction: 694 chars — Martinez et al. (2023)
  [P2] Methodology: 872 chars — Martinez et al. (2023)
  [P2] Results: 749 chars — Martinez et al. (2023)
  [P2] Discussion: 848 chars — Martinez et al. (2023)
  [P3] Abstract: 608 chars — Johansson et al. (2024)
  [P3] Introduction: 752 chars — Johansson et al. (2024)
  [P3] Methodology: 813 chars — Johansson et al. (2024)
  [P3] Results: 725 chars — Johansson et al. (2024)
  [P3] Discussion: 841 chars — Johansson et al. (2024)
  [P4] Abstract: 698 chars — Park et al. (2024)
  [P4] Introduction: 872 chars — Park et al. (2024)
  [P4] Methodology: 1014 chars — Park et al. (2024)
  [P4] Results: 876 char

In [5]:
# ── Chunk papers ──

class PaperChunk:
    """A chunk of a research paper with provenance tracking."""

    def __init__(self, text, section, chunk_idx, total_chunks):
        self.text = text
        self.paper_id = section.paper_id
        self.title = section.title
        self.authors = section.authors
        self.year = section.year
        self.venue = section.venue
        self.doi = section.doi
        self.heading = section.heading
        self.chunk_idx = chunk_idx
        self.total_chunks = total_chunks
        self.citation_short = section.citation
        self.citation_full = section.full_citation

        h = hashlib.md5(f"{self.paper_id}:{self.heading}:{chunk_idx}".encode()).hexdigest()[:8]
        self.chunk_id = f"{self.paper_id}-{h}"

    @property
    def metadata(self):
        return {
            "chunk_id": self.chunk_id,
            "paper_id": self.paper_id,
            "title": self.title,
            "authors": ", ".join(self.authors),
            "year": str(self.year),
            "venue": self.venue,
            "heading": self.heading,
        }

    @property
    def source_label(self):
        return (f"[{self.paper_id}] {self.citation_short}, "
                f"Section: {self.heading} "
                f"(chunk {self.chunk_idx+1}/{self.total_chunks})")


def chunk_section(section, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = section.text
    if len(text) <= chunk_size:
        return [PaperChunk(text, section, 0, 1)]
    parts = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        parts.append(text[start:end].strip())
        start += chunk_size - overlap
    return [PaperChunk(p, section, i, len(parts)) for i, p in enumerate(parts) if p]


all_chunks = []
for section in all_sections:
    all_chunks.extend(chunk_section(section))

print(f"Total chunks: {len(all_chunks)}\n")
per_paper = Counter(c.paper_id for c in all_chunks)
for pid, count in sorted(per_paper.items()):
    paper = next(p for p in PAPERS if p["paper_id"] == pid)
    print(f"  {pid}: {count} chunks — {paper['title'][:60]}")

Total chunks: 59

  P1: 10 chunks — Contextual Word Representations via Transformer Pre-training
  P2: 12 chunks — Efficient Fine-Tuning with Adapter Layers for NLU Tasks
  P3: 11 chunks — Cross-Lingual Transfer with Multilingual Transformers
  P4: 14 chunks — Robust NLU Under Distribution Shift: Adversarial and Domain 
  P5: 12 chunks — Knowledge-Enhanced Pre-training for Commonsense NLU


## Building the Research Index

In [6]:
# ── Research index ──

class ResearchIndex:
    """Vector index over research paper chunks."""

    def __init__(self, chunks):
        self.chunks = chunks
        self.texts = [c.text for c in chunks]
        self.chunk_map = {c.chunk_id: c for c in chunks}

        if USE_ST and USE_CHROMA:
            self._build_chroma()
        else:
            self._build_tfidf()

    def _build_chroma(self):
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("research")
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name="research", metadata={"hnsw:space": "cosine"}
        )
        embeddings = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [c.chunk_id for c in self.chunks]
        metas = [c.metadata for c in self.chunks]
        self.collection.add(ids=ids, embeddings=embeddings,
                            documents=self.texts, metadatas=metas)
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} chunks")

    def _build_tfidf(self):
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} chunks")

    def search(self, query, top_k=TOP_K, paper_filter=None):
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, paper_filter)
        return self._search_tfidf(query, top_k, paper_filter)

    def _search_chroma(self, query, top_k, paper_filter):
        where = {"paper_id": paper_filter} if paper_filter else None
        query_emb = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(top_k, self.collection.count()),
            where=where,
        )
        output = []
        for i, uid in enumerate(results["ids"][0]):
            chunk = self.chunk_map[uid]
            sim = 1.0 - results["distances"][0][i]
            output.append((chunk, sim))
        return output

    def _search_tfidf(self, query, top_k, paper_filter):
        candidates = []
        for i, c in enumerate(self.chunks):
            if paper_filter and c.paper_id != paper_filter:
                continue
            candidates.append(i)
        if not candidates:
            return []
        query_vec = self.vectorizer.transform([query])
        cand_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[candidates[i]], float(sims[i])) for i in top_idx]


research_index = ResearchIndex(all_chunks)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 59 chunks


## Baseline: Keyword Search

In [7]:
# ── Baseline ──

def keyword_search(query, chunks, top_k=TOP_K):
    q_terms = set(query.lower().split())
    scored = []
    for c in chunks:
        text_lower = c.text.lower()
        hits = sum(1 for t in q_terms if t in text_lower)
        scored.append((c, hits / max(len(q_terms), 1)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

test_q = "What accuracy does TransContext achieve on GLUE?"
baseline_r = keyword_search(test_q, all_chunks, top_k=3)

print(f"Query: {test_q}\n")
for c, score in baseline_r:
    print(f"  Score: {score:.2f} | {c.source_label}")
    print(f"  Preview: {c.text[:100]}...\n")

Query: What accuracy does TransContext achieve on GLUE?

  Score: 0.57 | [P5] Wang et al. (2024), Section: Results (chunk 1/2)
  Preview: CommonsenseQA: CST achieves 84.7%, compared to 72.1% (BERT-Large), 78.9% (RoBERTa-Large), and 80.3% ...

  Score: 0.43 | [P1] Chen et al. (2023), Section: Abstract (chunk 1/2)
  Preview: We present TransContext, a transformer-based language model pre-trained on 800GB of web text using m...

  Score: 0.43 | [P1] Chen et al. (2023), Section: Results (chunk 1/2)
  Preview: On GLUE benchmark, TransContext-Large achieves an average score of 92.4%, compared to 87.6% for BERT...



## Single-Hop Retrieval

First, let's verify that single-hop queries (answerable by one paper) work well.

In [8]:
# ── Single-hop queries ──

single_hop_queries = [
    "What is the dynamic masking schedule in TransContext?",
    "How much memory do adapters save compared to full fine-tuning?",
    "How does PolyGlot-XL perform on low-resource languages?",
    "What is the adversarial training approach in RobustNLU?",
    "What knowledge graphs does CommonsenseTransformer use?",
]

print("=== Single-Hop Retrieval ===\n")
for q in single_hop_queries:
    results = research_index.search(q, top_k=1)
    if results:
        chunk, score = results[0]
        print(f"Q: {q}")
        print(f"  [{score:.3f}] {chunk.source_label}")
        print(f"  {chunk.text[:120]}...\n")

=== Single-Hop Retrieval ===

Q: What is the dynamic masking schedule in TransContext?
  [0.543] [P1] Chen et al. (2023), Section: Methodology (chunk 1/2)
  TransContext uses the standard transformer encoder architecture. Pre-training uses two objectives: (1) Masked Language M...

Q: How much memory do adapters save compared to full fine-tuning?
  [0.629] [P2] Martinez et al. (2023), Section: Results (chunk 1/2)
  AdaptNLU achieves 90.8% on GLUE with only 3.6% additional parameters, compared to 92.1% for full fine-tuning (97.2% rela...

Q: How does PolyGlot-XL perform on low-resource languages?
  [0.690] [P3] Johansson et al. (2024), Section: Results (chunk 1/2)
  On XTREME, PolyGlot-XL achieves 82.3% average across all 40 languages, compared to 79.2% for XLM-RoBERTa-Large. Gains ar...

Q: What is the adversarial training approach in RobustNLU?
  [0.784] [P4] Park et al. (2024), Section: Methodology (chunk 1/3)
  RobustNLU extends the standard fine-tuning pipeline with three components

## Multi-Hop: Query Decomposition

Complex questions require **decomposition** into simpler sub-queries. In a
production system, an LLM would decompose queries. Here we implement a
rule-based decomposer that handles common research question patterns.

In [9]:
# ── Query decomposition ──

COMPARISON_PATTERNS = [
    (r"compare|comparison|versus|vs\.?|differ", "comparison"),
    (r"advantage|disadvantage|benefit|drawback|tradeoff", "tradeoff"),
    (r"how do .+ and .+", "comparison"),
    (r"relationship between .+ and .+", "relationship"),
]

def decompose_query(query):
    """Break a multi-hop query into sub-queries."""
    q_lower = query.lower()

    # Check if it's a comparison/multi-hop question
    is_multi = False
    query_type = "single"
    for pattern, qtype in COMPARISON_PATTERNS:
        if re.search(pattern, q_lower):
            is_multi = True
            query_type = qtype
            break

    # Also detect questions mentioning multiple concepts
    concept_groups = [
        (["adapter", "parameter-efficient", "fine-tuning efficiency"], "adapters"),
        (["pre-training", "pre-trained", "from scratch"], "pre-training"),
        (["multilingual", "cross-lingual", "low-resource language"], "multilingual"),
        (["robustness", "adversarial", "distribution shift", "ood"], "robustness"),
        (["commonsense", "knowledge graph", "reasoning"], "commonsense"),
        (["transformer", "attention", "architecture"], "architecture"),
        (["benchmark", "glue", "accuracy", "performance"], "performance"),
    ]

    found_concepts = []
    for keywords, concept in concept_groups:
        if any(kw in q_lower for kw in keywords):
            found_concepts.append(concept)

    if len(found_concepts) >= 2:
        is_multi = True
        query_type = "multi-concept"

    if not is_multi:
        return {"type": "single", "sub_queries": [query], "original": query}

    # Generate sub-queries
    sub_queries = [query]  # always include original
    for concept in found_concepts:
        sub_q = f"{concept} in NLU research"
        sub_queries.append(sub_q)

    # Add targeted sub-queries for comparisons
    if "compare" in q_lower or "vs" in q_lower:
        # split on comparison words
        parts = re.split(r'\bcompare\b|\bvs\.?\b|\bversus\b|\band\b', q_lower)
        for part in parts:
            part = part.strip(" ?.,")
            if len(part) > 10:
                sub_queries.append(part)

    return {"type": query_type, "sub_queries": list(dict.fromkeys(sub_queries)),
            "original": query}


# Demo
test_queries = [
    "What accuracy does TransContext achieve on GLUE?",
    "How do adapters compare to full fine-tuning for NLU tasks?",
    "How does adversarial training affect multilingual model robustness?",
    "What are the tradeoffs between knowledge-enhanced pre-training and standard pre-training?",
]

for q in test_queries:
    result = decompose_query(q)
    print(f"Q: {q}")
    print(f"  Type: {result['type']}")
    for i, sq in enumerate(result['sub_queries']):
        print(f"  Sub-query {i+1}: {sq}")
    print()

Q: What accuracy does TransContext achieve on GLUE?
  Type: single
  Sub-query 1: What accuracy does TransContext achieve on GLUE?

Q: How do adapters compare to full fine-tuning for NLU tasks?
  Type: comparison
  Sub-query 1: How do adapters compare to full fine-tuning for NLU tasks?
  Sub-query 2: adapters in NLU research
  Sub-query 3: how do adapters
  Sub-query 4: to full fine-tuning for nlu tasks

Q: How does adversarial training affect multilingual model robustness?
  Type: multi-concept
  Sub-query 1: How does adversarial training affect multilingual model robustness?
  Sub-query 2: multilingual in NLU research
  Sub-query 3: robustness in NLU research

Q: What are the tradeoffs between knowledge-enhanced pre-training and standard pre-training?
  Type: tradeoff
  Sub-query 1: What are the tradeoffs between knowledge-enhanced pre-training and standard pre-training?
  Sub-query 2: pre-training in NLU research



## Multi-Hop Retrieval Engine

The multi-hop retriever:
1. Decomposes the query into sub-queries
2. Retrieves evidence for each sub-query independently
3. De-duplicates and re-ranks results across sub-queries
4. Groups results by source paper
5. Identifies whether evidence is corroborating, complementary, or contradicting

In [10]:
# ── Multi-hop retriever ──

class MultiHopRetriever:
    """Multi-hop retrieval across multiple research papers."""

    def __init__(self, index):
        self.index = index

    def retrieve(self, query, top_k=TOP_K, per_sub_k=MULTI_HOP_TOP_K):
        decomposition = decompose_query(query)
        is_multi = decomposition["type"] != "single"

        # Retrieve for each sub-query
        all_results = {}
        sub_query_results = {}

        for sub_q in decomposition["sub_queries"]:
            results = self.index.search(sub_q, top_k=per_sub_k)
            sub_query_results[sub_q] = results
            for chunk, score in results:
                if chunk.chunk_id not in all_results:
                    all_results[chunk.chunk_id] = {"chunk": chunk, "score": score,
                                                    "sub_queries": [sub_q]}
                else:
                    # Boost score if chunk appears in multiple sub-query results
                    existing = all_results[chunk.chunk_id]
                    existing["score"] = max(existing["score"], score) * 1.1
                    existing["sub_queries"].append(sub_q)

        # Sort by final score
        ranked = sorted(all_results.values(), key=lambda x: x["score"], reverse=True)
        ranked = ranked[:top_k]

        # Group by paper
        paper_groups = defaultdict(list)
        for item in ranked:
            paper_groups[item["chunk"].paper_id].append(item)

        # Classify evidence relationships
        evidence_type = self._classify_evidence(ranked, is_multi)

        return {
            "query": query,
            "decomposition": decomposition,
            "results": ranked,
            "paper_groups": dict(paper_groups),
            "evidence_type": evidence_type,
            "is_multi_hop": is_multi,
            "papers_used": len(paper_groups),
        }

    def _classify_evidence(self, ranked, is_multi):
        if not is_multi or len(ranked) < 2:
            return "single_source"

        papers = set(r["chunk"].paper_id for r in ranked)
        if len(papers) == 1:
            return "single_source"

        # Check if multiple papers address similar topics (complementary)
        headings = [r["chunk"].heading for r in ranked]
        if len(set(headings)) > 1:
            return "complementary"

        return "corroborating"


retriever = MultiHopRetriever(research_index)
print("Multi-hop retriever ready.")

Multi-hop retriever ready.


In [11]:
# ── Multi-hop demo ──

def display_retrieval(result):
    """Display multi-hop retrieval results."""
    print("=" * 70)
    print(f"Q: {result['query']}")
    print(f"Retrieval type: {'MULTI-HOP' if result['is_multi_hop'] else 'SINGLE-HOP'}")
    print(f"Evidence type: {result['evidence_type']}")
    print(f"Papers used: {result['papers_used']}")
    print(f"Decomposition: {result['decomposition']['type']}")
    for i, sq in enumerate(result['decomposition']['sub_queries']):
        print(f"  Sub-query {i+1}: {sq}")
    print("-" * 70)

    for paper_id, items in result['paper_groups'].items():
        paper_title = items[0]["chunk"].title
        print(f"\n  Paper [{paper_id}]: {paper_title}")
        for item in items:
            c = item["chunk"]
            print(f"    [{item['score']:.3f}] {c.heading} "
                  f"(chunk {c.chunk_idx+1}/{c.total_chunks})")
            print(f"    Preview: {c.text[:100]}...")

    print("=" * 70)


# Single-hop query
r1 = retriever.retrieve("What masking strategy does TransContext use?")
display_retrieval(r1)

Q: What masking strategy does TransContext use?
Retrieval type: SINGLE-HOP
Evidence type: single_source
Papers used: 1
Decomposition: single
  Sub-query 1: What masking strategy does TransContext use?
----------------------------------------------------------------------

  Paper [P1]: Contextual Word Representations via Transformer Pre-training
    [0.610] Methodology (chunk 1/2)
    Preview: TransContext uses the standard transformer encoder architecture. Pre-training uses two objectives: (...
    [0.462] Abstract (chunk 1/2)
    Preview: We present TransContext, a transformer-based language model pre-trained on 800GB of web text using m...
    [0.433] Abstract (chunk 2/2)
    Preview: model's perplexity, leading to 15% faster convergence compared to static masking approaches like BER...


In [12]:
# ── Multi-hop: compare adapters vs full pre-training ──

r2 = retriever.retrieve(
    "How do adapters compare to full fine-tuning for NLU performance?"
)
display_retrieval(r2)

Q: How do adapters compare to full fine-tuning for NLU performance?
Retrieval type: MULTI-HOP
Evidence type: single_source
Papers used: 1
Decomposition: multi-concept
  Sub-query 1: How do adapters compare to full fine-tuning for NLU performance?
  Sub-query 2: adapters in NLU research
  Sub-query 3: performance in NLU research
  Sub-query 4: how do adapters
  Sub-query 5: to full fine-tuning for nlu performance
----------------------------------------------------------------------

  Paper [P2]: Efficient Fine-Tuning with Adapter Layers for NLU Tasks
    [0.701] Discussion (chunk 2/3)
    Preview: omain (e.g., biomedical NER drops to 89% of full fine-tuning performance). This suggests that adapte...
    [0.651] Methodology (chunk 1/3)
    Preview: AdaptNLU inserts adapter modules after the feed-forward layer in each transformer block. Each adapte...
    [0.650] Results (chunk 1/2)
    Preview: AdaptNLU achieves 90.8% on GLUE with only 3.6% additional parameters, compared to 92.1% for 

In [13]:
# ── Multi-hop: robustness + multilingual ──

r3 = retriever.retrieve(
    "How does adversarial training improve robustness, and does it apply "
    "to multilingual models?"
)
display_retrieval(r3)

Q: How does adversarial training improve robustness, and does it apply to multilingual models?
Retrieval type: MULTI-HOP
Evidence type: complementary
Papers used: 2
Decomposition: multi-concept
  Sub-query 1: How does adversarial training improve robustness, and does it apply to multilingual models?
  Sub-query 2: multilingual in NLU research
  Sub-query 3: robustness in NLU research
----------------------------------------------------------------------

  Paper [P4]: Robust NLU Under Distribution Shift: Adversarial and Domain Adaptation
    [0.659] Introduction (chunk 2/3)
    Preview: butions inevitably differ. Prior work on adversarial training (Madry et al., 2018) and domain adapta...
    [0.568] Discussion (chunk 1/3)
    Preview: Our results show that robustness can be achieved with minimal in-distribution performance sacrifice....
    [0.568] Methodology (chunk 1/3)
    Preview: RobustNLU extends the standard fine-tuning pipeline with three components: (1) Adversarial Training:.

In [14]:
# ── Multi-hop: knowledge + performance tradeoffs ──

r4 = retriever.retrieve(
    "What are the tradeoffs between knowledge-enhanced pre-training "
    "and standard transformer pre-training?"
)
display_retrieval(r4)

Q: What are the tradeoffs between knowledge-enhanced pre-training and standard transformer pre-training?
Retrieval type: MULTI-HOP
Evidence type: complementary
Papers used: 4
Decomposition: multi-concept
  Sub-query 1: What are the tradeoffs between knowledge-enhanced pre-training and standard transformer pre-training?
  Sub-query 2: pre-training in NLU research
  Sub-query 3: architecture in NLU research
----------------------------------------------------------------------

  Paper [P5]: Knowledge-Enhanced Pre-training for Commonsense NLU
    [0.547] Introduction (chunk 1/2)
    Preview: Despite achieving high scores on many NLU benchmarks, pre-trained language models often fail at task...
    [0.515] Abstract (chunk 1/2)
    Preview: Standard pre-trained language models struggle with tasks requiring commonsense reasoning, such as ph...

  Paper [P2]: Efficient Fine-Tuning with Adapter Layers for NLU Tasks
    [0.543] Discussion (chunk 2/3)
    Preview: omain (e.g., biomedical NER dr

## Full Research Librarian Pipeline

In [15]:
# ── Research Librarian ──

class ResearchLibrarian:
    """Multi-hop research Q&A with cross-paper evidence synthesis."""

    def __init__(self, index, papers):
        self.retriever = MultiHopRetriever(index)
        self.papers = {p["paper_id"]: p for p in papers}

    def ask(self, question, top_k=TOP_K, verbose=True):
        """Answer a research question with multi-paper evidence."""
        result = self.retriever.retrieve(question, top_k=top_k)
        answer = self._synthesize(question, result)
        citations = self._build_citations(result)
        confidence = self._confidence(result)

        output = {
            "question": question,
            "answer": answer,
            "citations": citations,
            "is_multi_hop": result["is_multi_hop"],
            "evidence_type": result["evidence_type"],
            "papers_used": result["papers_used"],
            "confidence": confidence,
        }

        if verbose:
            self._display(output, result)

        return output

    def _synthesize(self, question, result):
        parts = []
        q_terms = set(question.lower().split())

        for paper_id, items in result["paper_groups"].items():
            paper_evidence = []
            for item in items:
                chunk = item["chunk"]
                sentences = re.split(r'(?<=[.!?])\s+', chunk.text)
                relevant = []
                for sent in sentences:
                    has_number = bool(re.search(r'\d+\.?\d*%|\d+[MBK]\b', sent))
                    overlap = sum(1 for t in q_terms if t in sent.lower())
                    if (overlap > 0 or has_number) and len(sent) > 20:
                        relevant.append((sent, overlap + (1 if has_number else 0)))
                relevant.sort(key=lambda x: x[1], reverse=True)
                paper_evidence.extend([s for s, _ in relevant[:2]])

            if paper_evidence:
                citation = items[0]["chunk"].citation_short
                evidence_text = " ".join(paper_evidence[:3])
                parts.append(f"{evidence_text} [{citation}]")

        if parts:
            return "\n\n".join(parts)
        # Fallback
        top = result["results"][0]["chunk"]
        return f"{top.text[:300]}... [{top.citation_short}]"

    def _build_citations(self, result):
        seen = set()
        citations = []
        for paper_id, items in result["paper_groups"].items():
            if paper_id not in seen:
                seen.add(paper_id)
                chunk = items[0]["chunk"]
                citations.append({
                    "paper_id": paper_id,
                    "short": chunk.citation_short,
                    "full": chunk.citation_full,
                    "sections_used": list(set(i["chunk"].heading for i in items)),
                    "relevance": max(i["score"] for i in items),
                })
        return citations

    def _confidence(self, result):
        scores = [r["score"] for r in result["results"]]
        if not scores:
            return 0.0
        top = scores[0]
        avg = np.mean(scores)
        multi_boost = 0.05 if result["papers_used"] > 1 else 0
        conf = 0.6 * min(top / 0.6, 1.0) + 0.4 * min(avg / 0.4, 1.0) + multi_boost
        return round(min(conf, 1.0), 3)

    def _display(self, output, result):
        print("=" * 70)
        hop = "MULTI-HOP" if output["is_multi_hop"] else "SINGLE-HOP"
        print(f"Q: {output['question']}")
        print(f"   [{hop}] Evidence: {output['evidence_type']} | "
              f"Papers: {output['papers_used']}")
        print("=" * 70)

        print(f"\nAnswer:")
        for line in output["answer"].split("\n"):
            if line.strip():
                wrapped = textwrap.fill(line.strip(), width=66,
                                        initial_indent="  ", subsequent_indent="  ")
                print(wrapped)

        level = "HIGH" if output["confidence"] >= 0.7 else (
            "MEDIUM" if output["confidence"] >= 0.4 else "LOW")
        bar = "#" * int(output["confidence"] * 25)
        print(f"\nConfidence: {output['confidence']:.3f} ({level}) |{bar}|")

        print(f"\nCitations:")
        for i, cite in enumerate(output["citations"], 1):
            print(f"  [{i}] {cite['full']}")
            print(f"      Sections: {', '.join(cite['sections_used'])}")
            print(f"      Relevance: {cite['relevance']:.3f}")

        print("=" * 70)


librarian = ResearchLibrarian(research_index, PAPERS)
print("Research Librarian ready.")

Research Librarian ready.


## Query Demonstrations

In [16]:
# ── Single-hop query ──
librarian.ask("What is the dynamic masking schedule used in TransContext?")


Q: What is the dynamic masking schedule used in TransContext?
   [SINGLE-HOP] Evidence: single_source | Papers: 1

Answer:
  The dynamic masking schedule starts at 10% masking ratio and
  increases linearly to 25% over 500K training steps, with
  adjustments based on a moving average of validation perplexity
  computed every 10K steps. Pre-training uses two objectives: (1)
  Masked Language Modeling (MLM) with our novel dynamic masking
  schedule, and (2) Next Sentence Prediction (NSP). We propose
  dynamic masking scheduling, which adjusts the masking ratio from
  10% to 25% over the course of training based on validation
  perplexity. [Chen et al. (2023)]

Confidence: 0.948 (HIGH) |#######################|

Citations:
  [1] Chen, W., Liu, R., Patel, S.. "Contextual Word Representations via Transformer Pre-training." ACL 2023. DOI: 10.1234/acl.2023.001
      Sections: Introduction, Abstract, Methodology
      Relevance: 0.548


{'question': 'What is the dynamic masking schedule used in TransContext?',
 'answer': 'The dynamic masking schedule starts at 10% masking ratio and increases linearly to 25% over 500K training steps, with adjustments based on a moving average of validation perplexity computed every 10K steps. Pre-training uses two objectives: (1) Masked Language Modeling (MLM) with our novel dynamic masking schedule, and (2) Next Sentence Prediction (NSP). We propose dynamic masking scheduling, which adjusts the masking ratio from 10% to 25% over the course of training based on validation perplexity. [Chen et al. (2023)]',
 'citations': [{'paper_id': 'P1',
   'short': 'Chen et al. (2023)',
   'full': 'Chen, W., Liu, R., Patel, S.. "Contextual Word Representations via Transformer Pre-training." ACL 2023. DOI: 10.1234/acl.2023.001',
   'sections_used': ['Introduction', 'Abstract', 'Methodology'],
   'relevance': 0.5484879612922668}],
 'is_multi_hop': False,
 'evidence_type': 'single_source',
 'papers_use

In [17]:
# ── Multi-hop: compare efficiency approaches ──
librarian.ask("How do adapters compare to full fine-tuning in terms of "
              "performance and compute cost?")

Q: How do adapters compare to full fine-tuning in terms of performance and compute cost?
   [MULTI-HOP] Evidence: complementary | Papers: 2

Answer:
  omain (e.g., biomedical NER drops to 89% of full fine-tuning
  performance). Compared to prompt tuning approaches, adapters
  provide more stable training and better performance on smaller
  datasets (under 10K examples), likely because adapters modify
  intermediate representations while prompt tuning only modifies
  the input. AdaptNLU achieves 90.8% on GLUE with only 3.6%
  additional parameters, compared to 92.1% for full fine-tuning
  (97.2% relative performance). [Martinez et al. (2023)]
  racy compared to 64.1% for standard fine-tuning — a 14.2 point
  improvement. In-distribution performance: RobustNLU sacrifices
  only 0.8 points on GLUE (91.3% vs 92.1% for standard fine-
  tuning), a favorable tradeoff given the robustness gains. [Park
  et al. (2024)]

Confidence: 1.000 (HIGH) |#########################|

Citations:
  [1] Mart

{'question': 'How do adapters compare to full fine-tuning in terms of performance and compute cost?',
 'answer': 'omain (e.g., biomedical NER drops to 89% of full fine-tuning performance). Compared to prompt tuning approaches, adapters provide more stable training and better performance on smaller datasets (under 10K examples), likely because adapters modify intermediate representations while prompt tuning only modifies the input. AdaptNLU achieves 90.8% on GLUE with only 3.6% additional parameters, compared to 92.1% for full fine-tuning (97.2% relative performance). [Martinez et al. (2023)]\n\nracy compared to 64.1% for standard fine-tuning — a 14.2 point improvement. In-distribution performance: RobustNLU sacrifices only 0.8 points on GLUE (91.3% vs 92.1% for standard fine-tuning), a favorable tradeoff given the robustness gains. [Park et al. (2024)]',
 'citations': [{'paper_id': 'P2',
   'short': 'Martinez et al. (2023)',
   'full': 'Martinez, A., Kim, J., Thompson, E.. "Efficient F

In [18]:
# ── Multi-hop: cross-paper synthesis ──
librarian.ask("What pre-training strategies improve performance on low-resource "
              "languages and commonsense reasoning?")


Q: What pre-training strategies improve performance on low-resource languages and commonsense reasoning?
   [MULTI-HOP] Evidence: complementary | Papers: 2

Answer:
  These failures arise because commonsense knowledge is implicit
  in text and rarely stated explicitly. Despite achieving high
  scores on many NLU benchmarks, pre-trained language models often
  fail at tasks requiring basic commonsense. CST demonstrates that
  explicit knowledge integration substantially improves
  commonsense reasoning without degrading standard NLU
  performance. [Wang et al. (2024)]
  Multilingual models offer a path to NLP capabilities for low-
  resource languages through cross-lingual transfer — training on
  resource-rich languages and evaluating on others. Most NLP
  research focuses on English, leaving thousands of languages
  underserved. [Johansson et al. (2024)]

Confidence: 1.000 (HIGH) |#########################|

Citations:
  [1] Wang, H., Petrov, N., Al-Rashid, F.. "Knowledge-Enhanced Pr

{'question': 'What pre-training strategies improve performance on low-resource languages and commonsense reasoning?',
 'answer': 'These failures arise because commonsense knowledge is implicit in text and rarely stated explicitly. Despite achieving high scores on many NLU benchmarks, pre-trained language models often fail at tasks requiring basic commonsense. CST demonstrates that explicit knowledge integration substantially improves commonsense reasoning without degrading standard NLU performance. [Wang et al. (2024)]\n\nMultilingual models offer a path to NLP capabilities for low-resource languages through cross-lingual transfer — training on resource-rich languages and evaluating on others. Most NLP research focuses on English, leaving thousands of languages underserved. [Johansson et al. (2024)]',
 'citations': [{'paper_id': 'P5',
   'short': 'Wang et al. (2024)',
   'full': 'Wang, H., Petrov, N., Al-Rashid, F.. "Knowledge-Enhanced Pre-training for Commonsense NLU." ACL 2024. DOI: 

In [19]:
# ── Multi-hop: robustness landscape ──
librarian.ask("How do different papers address the problem of model robustness "
              "and distribution shift?")

Q: How do different papers address the problem of model robustness and distribution shift?
   [MULTI-HOP] Evidence: complementary | Papers: 2

Answer:
  We propose an integrated framework that addresses three aspects
  of robustness: adversarial robustness (resistance to input
  perturbations), domain robustness (generalization across
  domains), and calibration (reliable confidence estimates). Prior
  work on adversarial training (Madry et al., 2018) and domain
  adaptation (Ganin et al., 2016) has shown promise individually,
  but few studies combine these approaches for NLU tasks.
  Comparison with data augmentation baselines: random text
  augmentation (EDA, Wei & Zou 2019) achieves 80% distribution
  shift retention, compared to our 91%, suggesting that targeted
  adversarial perturbations are more effective than random
  augmentations. [Park et al. (2024)]
  The next day, was the cheese fresh or rotten?' requires
  understanding of food preservation. Despite achieving high
  scor

{'question': 'How do different papers address the problem of model robustness and distribution shift?',
 'answer': "We propose an integrated framework that addresses three aspects of robustness: adversarial robustness (resistance to input perturbations), domain robustness (generalization across domains), and calibration (reliable confidence estimates). Prior work on adversarial training (Madry et al., 2018) and domain adaptation (Ganin et al., 2016) has shown promise individually, but few studies combine these approaches for NLU tasks. Comparison with data augmentation baselines: random text augmentation (EDA, Wei & Zou 2019) achieves 80% distribution shift retention, compared to our 91%, suggesting that targeted adversarial perturbations are more effective than random augmentations. [Park et al. (2024)]\n\nThe next day, was the cheese fresh or rotten?' requires understanding of food preservation. Despite achieving high scores on many NLU benchmarks, pre-trained language models often f

In [20]:
# ── Multi-hop: architecture comparison ──
librarian.ask("Compare the transformer architectures and parameter counts "
              "across the papers in this corpus.")

Q: Compare the transformer architectures and parameter counts across the papers in this corpus.
   [MULTI-HOP] Evidence: complementary | Papers: 4

Answer:
  All three components are applied to a 340M parameter transformer
  backbone (same architecture as TransContext-Large). idence and
  actual accuracy on a held-out calibration set. [Park et al.
  (2024)]
  For a 12-layer model, the total adapter parameters are 1.18M,
  compared to 110M base parameters (1.07% overhead for Base, 3.6%
  for our recommended configuration with wider bottleneck).
  AdaptNLU inserts adapter modules after the feed-forward layer in
  each transformer block. ers reduce GPU memory requirements by
  60% and training time by 45%, making state-of-the-art NLU
  accessible to researchers with limited compute budgets.
  [Martinez et al. (2023)]
  We propose dynamic masking scheduling, which adjusts the masking
  ratio from 10% to 25% over the course of training based on
  validation perplexity. Our approach builds o

{'question': 'Compare the transformer architectures and parameter counts across the papers in this corpus.',
 'answer': 'All three components are applied to a 340M parameter transformer backbone (same architecture as TransContext-Large). idence and actual accuracy on a held-out calibration set. [Park et al. (2024)]\n\nFor a 12-layer model, the total adapter parameters are 1.18M, compared to 110M base parameters (1.07% overhead for Base, 3.6% for our recommended configuration with wider bottleneck). AdaptNLU inserts adapter modules after the feed-forward layer in each transformer block. ers reduce GPU memory requirements by 60% and training time by 45%, making state-of-the-art NLU accessible to researchers with limited compute budgets. [Martinez et al. (2023)]\n\nWe propose dynamic masking scheduling, which adjusts the masking ratio from 10% to 25% over the course of training based on validation perplexity. Our approach builds on the transformer architecture with 12 attention heads and 

In [21]:
# ── Multi-hop: benchmark comparison across papers ──
librarian.ask("What GLUE scores are reported across different papers, and "
              "which approach achieves the highest?")


Q: What GLUE scores are reported across different papers, and which approach achieves the highest?
   [MULTI-HOP] Evidence: complementary | Papers: 4

Answer:
  The Base model (110M parameters) achieves 89.7% on GLUE,
  demonstrating that dynamic masking benefits smaller models
  proportionally more than larger ones. On GLUE benchmark,
  TransContext-Large achieves an average score of 92.4%, compared
  to 87.6% for BERT-Large and 90.1% for RoBERTa-Large. The Base
  model (110M parameters) achieves 89.7% on GLUE, demons [Chen et
  al. (2023)]
  CommonsenseQA: CST achieves 84.7%, compared to 72.1% (BERT-
  Large), 78.9% (RoBERTa-Large), and 80.3% (our TransContext
  baseline). On standard NLU (GLUE), CST performs comparably to
  TransContext (92.0% vs 92.4%), showing that knowledge
  enhancement does not harm standard NLU. [Wang et al. (2024)]
  On adversarial benchmarks (AdvGLUE), RobustNLU achieves 78.3%
  accuracy compared to 64.1% for standard fine-tuning — a 14.2
  point improvemen

{'question': 'What GLUE scores are reported across different papers, and which approach achieves the highest?',
 'answer': 'The Base model (110M parameters) achieves 89.7% on GLUE, demonstrating that dynamic masking benefits smaller models proportionally more than larger ones. On GLUE benchmark, TransContext-Large achieves an average score of 92.4%, compared to 87.6% for BERT-Large and 90.1% for RoBERTa-Large. The Base model (110M parameters) achieves 89.7% on GLUE, demons [Chen et al. (2023)]\n\nCommonsenseQA: CST achieves 84.7%, compared to 72.1% (BERT-Large), 78.9% (RoBERTa-Large), and 80.3% (our TransContext baseline). On standard NLU (GLUE), CST performs comparably to TransContext (92.0% vs 92.4%), showing that knowledge enhancement does not harm standard NLU. [Wang et al. (2024)]\n\nOn adversarial benchmarks (AdvGLUE), RobustNLU achieves 78.3% accuracy compared to 64.1% for standard fine-tuning — a 14.2 point improvement. Results: Standard fine-tuning retains 76% of in-distributi

## Cross-Paper Evidence Map

Let's build a structured view of which papers address which topics —
a literature review in miniature.

In [22]:
# ── Evidence map: which papers cover which topics ──

research_topics = [
    "pre-training objectives",
    "parameter efficiency",
    "multilingual NLP",
    "adversarial robustness",
    "commonsense reasoning",
    "benchmark performance on GLUE",
    "computational cost and training efficiency",
    "knowledge graphs for NLU",
]

print("=== Research Evidence Map ===")
print(f"{'Topic':<40} {'Papers':>30}")
print("-" * 70)

topic_paper_map = {}
for topic in research_topics:
    results = research_index.search(topic, top_k=5)
    papers_found = {}
    for chunk, score in results:
        if score > 0.15:  # threshold for relevance
            if chunk.paper_id not in papers_found:
                papers_found[chunk.paper_id] = score
    topic_paper_map[topic] = papers_found
    paper_str = ", ".join(f"{pid}({s:.2f})" for pid, s in
                          sorted(papers_found.items(), key=lambda x: x[1], reverse=True))
    print(f"  {topic:<38} {paper_str}")

print("-" * 70)
print(f"\nPaper coverage: how many topics each paper addresses")
paper_coverage = Counter()
for topic, papers in topic_paper_map.items():
    for pid in papers:
        paper_coverage[pid] += 1
for pid, count in paper_coverage.most_common():
    paper = PAPERS[int(pid[1])-1]
    print(f"  {pid}: {count}/{len(research_topics)} topics — {paper['title'][:50]}")

=== Research Evidence Map ===
Topic                                                            Papers
----------------------------------------------------------------------
  pre-training objectives                P3(0.43), P2(0.42), P1(0.39), P5(0.38)
  parameter efficiency                   P2(0.34), P1(0.33), P4(0.33)
  multilingual NLP                       P3(0.65)


  adversarial robustness                 P4(0.70)


  commonsense reasoning                  P5(0.59)
  benchmark performance on GLUE          P1(0.55), P2(0.52)


  computational cost and training efficiency P5(0.44), P4(0.41), P2(0.38), P1(0.36)
  knowledge graphs for NLU               P5(0.48)
----------------------------------------------------------------------

Paper coverage: how many topics each paper addresses
  P2: 4/8 topics — Efficient Fine-Tuning with Adapter Layers for NLU 
  P1: 4/8 topics — Contextual Word Representations via Transformer Pr
  P5: 4/8 topics — Knowledge-Enhanced Pre-training for Commonsense NL
  P4: 3/8 topics — Robust NLU Under Distribution Shift: Adversarial a
  P3: 2/8 topics — Cross-Lingual Transfer with Multilingual Transform


In [23]:
# ── Cross-paper agreement analysis ──

def analyze_cross_paper(topic, index, top_k=3):
    """Analyze how multiple papers address the same topic."""
    results = index.search(topic, top_k=top_k * 2)

    paper_evidence = defaultdict(list)
    for chunk, score in results:
        if score > 0.1:
            paper_evidence[chunk.paper_id].append({
                "text": chunk.text[:200],
                "score": score,
                "heading": chunk.heading,
                "citation": chunk.citation_short,
            })

    print(f"Topic: \"{topic}\"")
    print(f"Papers addressing this: {len(paper_evidence)}\n")

    for pid, evidence in paper_evidence.items():
        best = max(evidence, key=lambda x: x["score"])
        print(f"  [{pid}] {best['citation']} (Section: {best['heading']})")
        print(f"  Relevance: {best['score']:.3f}")
        print(f"  Evidence: {best['text'][:150]}...")
        print()

    if len(paper_evidence) >= 2:
        print("  Relationship: COMPLEMENTARY — multiple papers provide different "
              "perspectives on this topic.")
    elif len(paper_evidence) == 1:
        print("  Relationship: SINGLE SOURCE — only one paper addresses this directly.")
    print("-" * 60)


analyze_cross_paper("GLUE benchmark performance comparison", research_index)
print()
analyze_cross_paper("computational cost of pre-training", research_index)

Topic: "GLUE benchmark performance comparison"
Papers addressing this: 2

  [P1] Chen et al. (2023) (Section: Results)
  Relevance: 0.534
  Evidence: On GLUE benchmark, TransContext-Large achieves an average score of 92.4%, compared to 87.6% for BERT-Large and 90.1% for RoBERTa-Large. On SuperGLUE, ...

  [P2] Martinez et al. (2023) (Section: Results)
  Relevance: 0.518
  Evidence: AdaptNLU achieves 90.8% on GLUE with only 3.6% additional parameters, compared to 92.1% for full fine-tuning (97.2% relative performance). On SuperGLU...

  Relationship: COMPLEMENTARY — multiple papers provide different perspectives on this topic.
------------------------------------------------------------

Topic: "computational cost of pre-training"
Papers addressing this: 4

  [P2] Martinez et al. (2023) (Section: Introduction)
  Relevance: 0.495
  Evidence: The trend toward larger pre-trained models creates a tension: bigger models achieve better performance but are increasingly expensive to fine-tune. 

## Evaluation

In [24]:
# ── Evaluation ──

eval_set = [
    # Single-hop queries
    {"query": "Dynamic masking schedule in TransContext",
     "expected_papers": ["P1"], "hop_type": "single"},
    {"query": "Adapter parameter overhead percentage",
     "expected_papers": ["P2"], "hop_type": "single"},
    {"query": "PolyGlot-XL XTREME benchmark score",
     "expected_papers": ["P3"], "hop_type": "single"},
    {"query": "NLU-Shift benchmark and distribution shift",
     "expected_papers": ["P4"], "hop_type": "single"},
    {"query": "CommonsenseQA accuracy with knowledge graphs",
     "expected_papers": ["P5"], "hop_type": "single"},
    # Multi-hop queries
    {"query": "Compare pre-training compute costs across papers",
     "expected_papers": ["P1", "P3", "P5"], "hop_type": "multi"},
    {"query": "How do efficiency and robustness tradeoff in NLU",
     "expected_papers": ["P2", "P4"], "hop_type": "multi"},
    {"query": "Approaches to improving low-resource and commonsense NLU",
     "expected_papers": ["P3", "P5"], "hop_type": "multi"},
]

baseline_hits = 0
semantic_hits = 0
multi_hop_paper_recall = []

print(f"{'Query':<50} {'B-Hit':^6} {'S-Hit':^6} {'Papers':^10}")
print("-" * 75)

for e in eval_set:
    q = e["query"]
    expected = set(e["expected_papers"])

    # Baseline
    b = keyword_search(q, all_chunks, top_k=3)
    b_papers = set(c.paper_id for c, _ in b[:3])
    b_hit = bool(expected & b_papers)
    baseline_hits += int(b_hit)

    # Semantic
    s = research_index.search(q, top_k=3)
    s_papers = set(c.paper_id for c, _ in s[:3])
    s_hit = bool(expected & s_papers)
    semantic_hits += int(s_hit)

    # For multi-hop, check paper recall (how many expected papers found)
    if e["hop_type"] == "multi":
        recall = len(expected & s_papers) / len(expected)
        multi_hop_paper_recall.append(recall)

    print(f"  {q:<48} {'HIT' if b_hit else 'MISS':^6} "
          f"{'HIT' if s_hit else 'MISS':^6} {','.join(sorted(s_papers)):^10}")

n = len(eval_set)
print("-" * 75)
print(f"  {'ACCURACY':<48} {baseline_hits/n:^6.1%} {semantic_hits/n:^6.1%}")
if multi_hop_paper_recall:
    avg_recall = np.mean(multi_hop_paper_recall)
    print(f"  {'Multi-hop paper recall':<48} {'':^6} {avg_recall:^6.1%}")

Query                                              B-Hit  S-Hit    Papers  
---------------------------------------------------------------------------
  Dynamic masking schedule in TransContext          HIT    HIT       P1    
  Adapter parameter overhead percentage             HIT    HIT     P2,P4   


  PolyGlot-XL XTREME benchmark score                HIT    HIT       P3    


  NLU-Shift benchmark and distribution shift        HIT    HIT       P4    
  CommonsenseQA accuracy with knowledge graphs      HIT    HIT       P5    
  Compare pre-training compute costs across papers  HIT    HIT    P2,P3,P5 


  How do efficiency and robustness tradeoff in NLU  HIT    HIT     P2,P4   


  Approaches to improving low-resource and commonsense NLU  HIT    HIT     P3,P5   
---------------------------------------------------------------------------
  ACCURACY                                         100.0% 100.0%
  Multi-hop paper recall                                  88.9% 


## Error Analysis

In [25]:
# ── Error analysis ──

edge_cases = [
    "What datasets were used for pre-training?",          # Multiple papers
    "Which paper has the best GLUE score?",                # Requires comparison
    "What are the ethical implications of NLU research?",  # Not covered
    "How does GPT compare to these approaches?",           # External reference
]

print("=== Error Analysis: Edge Cases ===\n")
for q in edge_cases:
    result = retriever.retrieve(q, top_k=3)
    top_score = result["results"][0]["score"] if result["results"] else 0
    papers = set(r["chunk"].paper_id for r in result["results"])

    print(f"Q: \"{q}\"")
    print(f"  Top score: {top_score:.3f} | Papers: {', '.join(sorted(papers))}")
    print(f"  Multi-hop: {result['is_multi_hop']} | Type: {result['evidence_type']}")
    if top_score < 0.2:
        print(f"  -> WARNING: Low confidence — question may be out of scope")
    print()

=== Error Analysis: Edge Cases ===



Q: "What datasets were used for pre-training?"
  Top score: 0.476 | Papers: P2, P3, P5
  Multi-hop: False | Type: single_source

Q: "Which paper has the best GLUE score?"
  Top score: 0.408 | Papers: P1, P2
  Multi-hop: False | Type: single_source

Q: "What are the ethical implications of NLU research?"
  Top score: 0.328 | Papers: P4, P5
  Multi-hop: False | Type: single_source



Q: "How does GPT compare to these approaches?"
  Top score: 0.327 | Papers: P2, P5
  Multi-hop: True | Type: complementary



In [26]:
# ── Export metrics ──

metrics = {
    "project": "Research Librarian",
    "corpus": {
        "papers": len(PAPERS),
        "sections": len(all_sections),
        "chunks": len(all_chunks),
        "chunks_per_paper": dict(Counter(c.paper_id for c in all_chunks)),
    },
    "retrieval_backend": research_index.backend,
    "evaluation": {
        "total_queries": len(eval_set),
        "baseline_accuracy": baseline_hits / len(eval_set),
        "semantic_accuracy": semantic_hits / len(eval_set),
        "multi_hop_paper_recall": float(np.mean(multi_hop_paper_recall))
            if multi_hop_paper_recall else None,
    },
    "config": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "top_k": TOP_K,
        "multi_hop_per_sub_k": MULTI_HOP_TOP_K,
        "embedding_model": EMBEDDING_MODEL if USE_ST else "TF-IDF",
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\15_Research_Librarian\metrics.json
{
  "project": "Research Librarian",
  "corpus": {
    "papers": 5,
    "sections": 25,
    "chunks": 59,
    "chunks_per_paper": {
      "P1": 10,
      "P2": 12,
      "P3": 11,
      "P4": 14,
      "P5": 12
    }
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "total_queries": 8,
    "baseline_accuracy": 1.0,
    "semantic_accuracy": 1.0,
    "multi_hop_paper_recall": 0.8888888888888888
  },
  "config": {
    "chunk_size": 500,
    "chunk_overlap": 80,
    "top_k": 5,
    "multi_hop_per_sub_k": 3,
    "embedding_model": "all-MiniLM-L6-v2"
  }
}


## Limitations

1. **No LLM for synthesis** — our answer synthesis uses keyword matching, not
   semantic understanding. A real Research Librarian would use GPT-4 or similar
   to generate coherent multi-source summaries.

2. **Rule-based query decomposition** — in production, an LLM decomposes queries
   contextually. Our rule-based approach misses many multi-hop patterns.

3. **No iterative retrieval** — true multi-hop may need multiple rounds:
   retrieve → reason → retrieve again with refined queries. Our system
   does one round of parallel retrieval.

4. **Small corpus** — 5 papers is far from realistic. Real literature corpora
   have thousands of papers, requiring scalable indexing.

5. **No citation parsing** — papers cite each other. A full system would parse
   citation graphs to find related work automatically.

6. **No contradiction detection** — identifying genuine disagreements between
   papers requires deeper NLU than keyword overlap.

7. **Synthetic papers** — our corpus is fictional. Real papers have more complex
   structure, notation, and cross-references.

## Common Mistakes

1. **Treating multi-hop as single-hop** — returning only the single best chunk
   for a comparison question. Always check if the question requires multiple
   sources before deciding the retrieval strategy.

2. **Mixing evidence periods** — citing an older paper's results alongside a
   newer paper without noting the time difference. Always include publication
   year in citations.

3. **Redundant retrieval** — returning multiple chunks from the same section
   of the same paper. De-duplicate by paper and section to maximize coverage.

4. **Ignoring negative evidence** — only finding papers that support a claim.
   A good Research Librarian also surfaces contradicting evidence.

5. **Over-decomposition** — splitting a simple question into too many sub-queries
   introduces noise. Only decompose when the query genuinely requires it.

6. **Citation fabrication** — synthesizing an answer that claims Paper X says
   something it doesn't. Every claim in the answer must trace directly to
   retrieved text, not inferred or hallucinated content.

7. **Conflating "not found" with "not true"** — if no paper discusses a topic,
   the system should say "no evidence found," not "this is false."

## Mini Challenge

### Exercise 1: Add a 6th Paper
Create a paper about "Vision-Language Pre-training" with sections on
architecture, multimodal fusion, and benchmark results. Re-index and test
queries that bridge NLP and vision.

### Exercise 2: Iterative Multi-Hop
Implement 2-round retrieval: retrieve once, extract key terms from results,
then retrieve again with refined queries. Compare this to single-round.

### Exercise 3: Citation Graph
Add a `references` field to each paper listing which other papers it cites.
Build a function that, given a paper, finds all cited papers in the corpus
and shows their relationships.

### Exercise 4: Consensus Detection
For a given topic, retrieve evidence from all papers and classify:
- Which papers agree?
- Which papers add complementary evidence?
- Which papers contradict each other?

### Exercise 5: Literature Review Generator
Given a topic, automatically generate a structured literature review paragraph
that synthesizes evidence from multiple papers with proper citations.

## Production Considerations

1. **Semantic Scholar / arXiv API** — use APIs to ingest real papers
   (metadata, abstracts, full text when available).

2. **PDF Extraction** — use GROBID, ScienceBeam, or Nougat for structured
   extraction of research papers into sections, tables, and figures.

3. **Citation Graph** — build a graph database (Neo4j) of paper citations
   for graph-based retrieval ("find papers cited by X that discuss Y").

4. **LLM-Powered Decomposition** — use GPT-4 or Qwen3-Instruct for
   intelligent query decomposition and answer synthesis.

5. **Iterative Retrieval** — implement retrieve-read-retrieve loops where
   the LLM reasons about partial answers and generates follow-up queries.

6. **Scalability** — for large corpora (100K+ papers), use approximate
   nearest neighbor search (Faiss, ScaNN) with metadata pre-filtering.

7. **User Interface** — Streamlit or Gradio interface with paper browsing,
   evidence highlighting, and citation management.

8. **Claim Verification** — cross-reference extracted claims against the
   original paper text to prevent hallucination in synthesized answers.

## How to Improve This Project

- **LLM synthesis** — use an LLM to generate coherent multi-source summaries
  instead of keyword-based extraction.
- **Iterative retrieval** — implement multi-round retrieve-reason-retrieve
  for genuinely complex multi-hop questions.
- **Fine-tuned embeddings** — train on academic text (SciBERT, SPECTER2) for
  better scientific document retrieval.
- **Table and figure extraction** — parse tables from papers and treat them
  as structured data for numerical queries.
- **Automatic related work** — given a new paper, automatically find the
  most relevant prior work in the corpus.
- **Contradiction detection** — train a classifier to identify when two
  passages present conflicting findings.

## Key Takeaways

1. **Multi-hop retrieval requires query decomposition** — break complex
   questions into simpler sub-queries that single-hop retrieval can handle.

2. **De-duplication across sub-queries is essential** — without it,
   the same chunk appears multiple times, wasting the top-K budget.

3. **Paper-level grouping reveals evidence structure** — grouping results
   by source paper shows how many independent sources support the answer.

4. **Evidence types matter** — distinguishing corroborating, complementary,
   and contradicting evidence is critical for research Q&A.

5. **Citations must be exact** — every claim traces to a specific paper,
   section, and chunk. Fabricated citations undermine trust.

6. **Single-hop baselines are necessary** — always verify that your system
   handles simple questions well before adding multi-hop complexity.